# CARES-PARETO-003 — targeted class-pair routing (real proposed-method ablation)

**One Kaggle cell. No retraining.** Automatically restore the previous SHA256-verified frozen SCV and LightGBM checkpoints plus all six already-trained out-of-fold model results from an **unpublished GitHub draft release**, and evaluate a new class-pair-specific correction. Kaggle Internet ON; authorize existing private GitHub `pushRare` secret. CPU only. It does **not** require the UNSW CSV because all data used are frozen training-derived OOF probabilities and the fixed validation predictions.

**Scientific safeguards:** Calibration on 70,164 original development-training OOF examples, audit on independent 70,164 OOF examples. Any rulebook must improve BOTH accuracy AND ten-class macro-F1, preserve DoS/Analysis/Backdoor recall within 2 percentage points, and pass the audit—otherwise the new method falls back to the previous model. All thresholds selected before looking at original internal-validation labels. The official test is untouched.

**Novelty is unverified:** Class-pair routing and out-of-fold decision calibration build on existing selective classification ideas. First obtain repeatable positive evidence; only then claim a novel mechanism with supporting 2024–2026 related-work analysis.

**After running:** Send the `CARES-PARETO-003 TRAIN-DERIVED RISK ROUTING COMPLETE` output. This version writes a complete measured results directory; the final step automatically archives model predictions privately and syncs verified aggregate metrics to GitHub.

In [ ]:
# CARES-PARETO-003: one clean Kaggle cell; no re-training required.
from pathlib import Path
import hashlib, tarfile, json, os, subprocess, sys
import requests
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-cares-pareto")
BASE=Path("/kaggle/working/research_outputs")
CARES=BASE/"CARES-OOF-002-S42"
RUN=BASE/"CARES-PARETO-003-S42"
ASSET_ID=655688241
EXPECTED_ARCHIVE="14cbb4fa56b7983d05551470bd2ba1df0e5c44ef0d3bd016337e2696822ad00e"
ARCHIVE=Path("/kaggle/working/CARES-OOF-002-S42_FULL_ARTIFACTS.tar")

def digest(path):
    h=hashlib.sha256()
    with open(path,"rb") as file:
        for block in iter(lambda:file.read(4*1024*1024),b""):
            h.update(block)
    return h.hexdigest()

token=UserSecretsClient().get_secret("pushRare")
assert token, "Enable the existing Kaggle secret pushRare for this notebook"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
print("FIXED SOURCE COMMIT:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip(),flush=True)

# Restore our original reference and two baseline predictions as genuine
# hash-pinned private artifacts, with absolutely NO model refitting.
for name,archive_name in (
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),
    ("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")
):
    subprocess.run([sys.executable,str(REPO/"scripts"/name),
        "--base",str(BASE),
        "--archive",str(Path("/kaggle/working")/archive_name)],
        cwd=str(REPO),env=env,check=True)

# Restore the six ALREADY TRAINED OOF checkpoints from last experiment.
NEEDED=CARES/"train_OOF_probabilities_PRIVATE.npz"
EXPECTED_OOF="32f634645ac98d54d7bdf50513d6421d8efdd555230939e5f34e0482e1d5a36c"
if not NEEDED.is_file() or digest(NEEDED)!=EXPECTED_OOF:
    if not ARCHIVE.is_file() or digest(ARCHIVE)!=EXPECTED_ARCHIVE:
        url=f"https://api.github.com/repos/itsCodeBakery/rare-attack-spatiotemporal-ids/releases/assets/{ASSET_ID}"
        response=requests.get(url,headers={
            "Authorization":"Bearer "+token,
            "Accept":"application/octet-stream",
            "X-GitHub-Api-Version":"2022-11-28"},timeout=180)
        response.raise_for_status()
        if "json" in response.headers.get("Content-Type",""):
            raise RuntimeError("GitHub draft returned metadata, not model archive bytes")
        ARCHIVE.write_bytes(response.content)
        assert digest(ARCHIVE)==EXPECTED_ARCHIVE, "CARES full archive SHA256 mismatch"
    CARES.mkdir(parents=True,exist_ok=True)
    with tarfile.open(ARCHIVE,"r") as tar:
        catalogue=json.load(tar.extractfile("PRIVATE_ARTIFACT_CATALOGUE.json"))
        assert catalogue["experiment_id"]=="CARES-OOF-002-S42"
        assert catalogue["official_test_used"] is False
        for item in catalogue["files"]:
            rel=Path(item["path"])
            assert rel.parts[0]=="run" and ".." not in rel.parts
            member=tar.getmember(str(rel))
            assert member.isfile() and not member.issym() and not member.islnk()
            dest=CARES/Path(*rel.parts[1:])
            dest.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(member) as src, dest.open("wb") as f:
                while True:
                    block=src.read(4*1024*1024)
                    if not block:break
                    f.write(block)
            assert digest(dest)==item["sha256"],f"Private restore checksum failed: {rel}"
assert digest(NEEDED)==EXPECTED_OOF
print("SIX FROZEN OOF MODELS RESTORED; NO RETRAINING",flush=True)

subprocess.run([sys.executable,str(REPO/"scripts/run_cares_pareto003.py"),
    "--input",str(CARES),
    "--baseline",str(BASE/"CTRL-TAB-001-S42"),
    "--config",str(REPO/"configs/cares_pareto003_seed73.json"),
    "--output",str(RUN)],cwd=str(REPO),env=env,check=True)

print("TRUE TRAIN-DERIVED PARETO ROUTING RESULTS:")
print((RUN/"summary_scores.csv").read_text(),flush=True)
print("Training-only gatebook:",(RUN/"protocol.json").read_text()[:1200])
print("Experiment preserved locally at",RUN)
# Archive the new genuine results immediately; no new model fitting.
subprocess.run([sys.executable,str(REPO/"scripts/archive_cares_pareto003.py"),
    "--run",str(RUN),
    "--archive","/kaggle/working/CARES-PARETO-003-S42_FULL_ARTIFACTS.tar"],
    cwd=str(REPO),env=env,check=True)
print("CARES-PARETO-003 ALL RESULTS ARCHIVED ON GITHUB",flush=True)
